# 01 - Problem formulation and EDA
**AIE231 Neural Networks | Chest X-Ray Images (Pneumonia)** | Milestone 1 (spec section 3.1)

**How to run on Colab:** upload the project folder (or zip) so that `src/` sits next to `notebooks/`, set Runtime to any (GPU not needed for EDA), then Run all.

In [ ]:
# Colab only: mount Drive so `src/` is importable. Adjust the path if you put the folder elsewhere.
import os
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/aie231-project/notebooks

## 1. Problem statement (spec 3.1)
**User:** a radiology team in a paediatric clinic that wants a *screening aid* that flags chest X-rays that may show pneumonia, so a doctor looks at them first.
**Prediction:** given one chest X-ray image, predict `NORMAL` (0) or `PNEUMONIA` (1).
**Cost of errors:** a *false negative* (sick child called normal) can delay treatment, which is dangerous. A *false positive* (healthy child flagged) costs a doctor a second look. So false negatives cost much more.

**Technical definition**
| Item | Choice | Why |
|---|---|---|
| Input | RGB image, 224 x 224 x 3, pixels scaled to [0,1] | VGG16/ResNet-50 expect 224 x 224 x 3 |
| Output | 1 unit, sigmoid = P(pneumonia) | binary task |
| Loss | binary cross-entropy (optionally class-weighted) | matches sigmoid output |
| **Primary metric** | **Recall of the PNEUMONIA class** | catches sick cases; accuracy is misleading when classes are imbalanced |
| Secondary | precision, F1, macro-F1, ROC-AUC, confusion matrix | a model can get recall 1.0 by predicting "pneumonia" for everything, so we also watch precision/specificity |

*Limitations statement (required for health data):* this is a course project. It is not a diagnostic tool and must not be used for real medical decisions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so `from src.data import ...` works
!pip -q install kagglehub
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from src.data import download, find_data_root, build_inventory, CLASSES
pd.set_option('display.width', 160)
SEED = 42

## 2. Load the data
`kagglehub` downloads the public dataset (if Colab asks for Kaggle credentials, add `KAGGLE_USERNAME` / `KAGGLE_KEY` under Colab Secrets, or upload kaggle.json). `build_inventory` walks every file once and records label, original split, image size and an MD5 hash.

In [ ]:
base = download()
root = find_data_root(base)
print('Data root:', root)
df = build_inventory(root)          # takes ~1-3 min (reads every image header + file bytes)
df.to_csv('inventory.csv', index=False)
print(len(df), 'images')
df.head()

## 3. Class distribution (the number we did NOT know before running this)
We count per original split and overall. The spec requires us to *measure* the imbalance before handling it.

In [ ]:
tab = df.pivot_table(index='orig_split', columns='label_name', values='path', aggfunc='count', margins=True)
display(tab)
tot = df['label_name'].value_counts()
print('Overall pneumonia : normal =', round(tot['PNEUMONIA']/tot['NORMAL'], 2), ': 1')
print('Pneumonia share  =', round(100*tot['PNEUMONIA']/len(df), 1), '%')
print('Majority-class accuracy (always say PNEUMONIA) =', round(tot.max()/len(df), 4))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
tab.drop('All').drop(columns='All').plot.bar(ax=ax[0], rot=0, title='Images per original split and class')
df['subtype'].value_counts().plot.bar(ax=ax[1], rot=0, title='Pneumonia subtype (from file names)')
plt.tight_layout(); plt.show()

**Read this:** if pneumonia is ~70-75% of images, a model that always answers "pneumonia" scores ~73% accuracy and 100% recall on pneumonia but **0% recall on Normal**. That is why we do not use accuracy alone. Note also that the official `val/` folder is tiny and the official `test/` folder has a *different* class ratio from `train/`: we will rebuild our own split in Milestone 2.

## 4. Duplicates and patient overlap (data quality)

In [ ]:
dup_groups = df[df.duplicated('md5', keep=False)]
print('Images whose bytes are identical to another image:', len(dup_groups), 'in', dup_groups['md5'].nunique(), 'groups')
# Do duplicates cross the original train/test boundary? (that would be leakage)
cross = dup_groups.groupby('md5')['orig_split'].nunique()
print('Duplicate groups that span more than one original split:', int((cross > 1).sum()))
pat = df.dropna(subset=['patient_id'])
print('Pneumonia images with a patient id:', len(pat), '| distinct patients:', pat['patient_id'].nunique())
print('Patients with >1 image:', int((pat.groupby('patient_id').size() > 1).sum()))
pcross = pat.groupby('patient_id')['orig_split'].nunique()
print('Patients that appear in more than one original split:', int((pcross > 1).sum()))
print('NORMAL images have no patient id in their file names, so we cannot group them by patient (a stated limitation).')

## 5. Image size and quality (the 'real-world mess')

In [ ]:
print(df[['width','height','megapixels','aspect','kb']].describe().round(2))
print('\nColour modes:\n', df['mode'].value_counts())
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for c, col in zip(CLASSES, ['tab:blue', 'tab:red']):
    s = df[df.label_name == c]
    ax[0].scatter(s.width, s.height, s=4, alpha=.3, label=c, c=col)
    ax[1].hist(s.aspect, bins=40, alpha=.5, label=c, color=col)
    ax[2].hist(s.kb, bins=40, alpha=.5, label=c, color=col)
ax[0].set(xlabel='width px', ylabel='height px', title='Image dimensions'); ax[1].set(xlabel='width/height', title='Aspect ratio'); ax[2].set(xlabel='file KB', title='File size')
for a in ax: a.legend()
plt.tight_layout(); plt.show()

**Why it matters:** sizes vary a lot, so every image must be resized to a fixed shape. Size/file-size differences that correlate with the label are a *shortcut risk*: the network could learn "big file = pneumonia" instead of lung patterns. We keep it in mind for error analysis.

## 6. Sample images per class

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for r, c in enumerate(CLASSES):
    s = df[df.label_name == c].sample(6, random_state=SEED)
    for a, (_, row) in zip(axes[r], s.iterrows()):
        a.imshow(Image.open(row.path).convert('L'), cmap='gray'); a.axis('off')
        a.set_title(f"{c}\n{row.width}x{row.height}", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Pixel statistics and outliers

In [ ]:
samp = df.groupby('label_name').sample(150, random_state=SEED)
def stats(p):
    a = np.asarray(Image.open(p).convert('L'), dtype='float32') / 255
    return a.mean(), a.std()
st = np.array([stats(p) for p in samp.path])
samp = samp.assign(mean_px=st[:,0], std_px=st[:,1])
print(samp.groupby('label_name')[['mean_px','std_px']].describe().round(3).T)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for c in CLASSES:
    s = samp[samp.label_name == c]
    ax[0].hist(s.mean_px, bins=25, alpha=.5, label=c); ax[1].hist(s.std_px, bins=25, alpha=.5, label=c)
ax[0].set(title='Mean brightness'); ax[1].set(title='Contrast (std)'); ax[0].legend()
plt.tight_layout(); plt.show()
# extreme images = candidates for quality outliers
print(samp.sort_values('mean_px').iloc[[0,1,-2,-1]][['path','label_name','mean_px','std_px']])

## 8. Findings -> modelling decisions (required at the end of the EDA)
Fill the numbers from the outputs above, then keep this table; it goes into the report (section 3).

| # | Finding | Decision |
|---|---|---|
| 1 | Pneumonia is the majority class (ratio = *see section 3*) | Primary metric = recall on pneumonia; report per-class P/R/F1; use class weights and/or augmentation of Normal; show effect on per-class recall |
| 2 | Official val has only 16 images; official test ratio differs from train | Merge all images and make our own stratified 70/15/15 split with a fixed seed |
| 3 | Duplicate / same-patient images exist (*see section 4*) | Remove exact duplicates first; group by patient id where available so a patient never spans splits; state the Normal-class limitation |
| 4 | Variable image sizes, some grey-scale only | Resize to 224x224, replicate to 3 channels, scale pixels to [0,1] (VGG16/ResNet need their own preprocess_input in transfer models) |
| 5 | Size / file-size / brightness differ between classes (*see sections 5, 7*) | Augmentation (flip is NOT used for X-rays by default; use small rotation, zoom, brightness/contrast); check shortcuts in error analysis |
| 6 | Majority-class baseline already reaches high accuracy | Never report accuracy alone; baselines must be beaten on recall AND specificity |